# Logarithmic Algorithms: Binary Search, Divide and Conquer, $O(\log n)$

This notebook explains why the logarithm is the most important function in the analysis of algorithms. Whenever an algorithm **halves** its problem at each step, the number of steps is $\log_2 n$; whenever it splits a problem in two and does linear work to combine the halves, the cost is $n \log_2 n$. We measure both empirically and see the logarithm appear in the counts.

This is notebook 04 of module 06 (**Applications**) in the **Learning Logarithms** series.

## 1. Introduction

A phone book lists $10^9$ names in alphabetical order. How long does it take to find one?

- **Read from the start** (linear search): in the worst case $10^9$ comparisons, on average half a billion. At a comparison per second, that is about $16$ years of reading in the average case.
- **Open in the middle** (binary search): compare with the middle name. If yours comes earlier, throw away the second half; otherwise throw away the first half. Repeat.

Each comparison halves what is left: $10^9 \to 5 \times 10^8 \to 2.5 \times 10^8 \to \dots$. How many halvings until one name is left? We need the $k$ with $2^k \ge 10^9$, which is

$$
k = \lceil \log_2 10^9 \rceil = \lceil 9 \log_2 10 \rceil = \lceil 29.9 \rceil = 30.
$$

**Thirty comparisons instead of a billion.** That is the logarithm as "the number of times you can halve something", and it is why databases, search engines, compilers and operating systems are built around halving.

**Prerequisites**

- Binary logarithms and change of base — [Change of base](../00_foundations/04_change_of_base_and_common_bases.ipynb).
- Comparing growth rates $\ln n \ll n \ll n \log_2 n \ll n^2$ — [Growth rates and asymptotics](../02_graphs_equations_and_scales/04_growth_rates_and_asymptotics.ipynb).
- "How many questions": bits and $\log_2$ — [Bits, digits and counting](../05_information_and_probability/01_bits_digits_and_counting.ipynb).
- Stirling's approximation of $\ln n!$, for the sorting lower bound in exercise 7 — [Stirling's approximation](../04_logarithms_in_calculus/04_stirlings_approximation.ipynb).

After completing this notebook you will be able to:

- Explain why repeated halving takes $\lceil \log_2 n \rceil$ steps, and why the base of the logarithm does not matter in $O(\log n)$.
- Implement binary search with a comparison counter and verify the bound $\lfloor \log_2 n \rfloor + 1$.
- Solve the divide-and-conquer recurrence $T(n) = 2T(n/2) + n$ and connect it to merge sort.
- Compute $x^n$ with about $\log_2 n$ multiplications by repeated squaring.
- Relate the height of a balanced binary tree to $\log_2 n$.

## 2. Theory

### 2.1 Repeated halving

Start with $n$ items and keep half after each step. After $k$ steps at most $n / 2^k$ remain (rounded up). We are done when one item remains:

$$
\frac{n}{2^k} \le 1 \quad\Longleftrightarrow\quad 2^k \ge n \quad\Longleftrightarrow\quad k \ge \log_2 n.
$$

So repeated halving takes $\lceil \log_2 n \rceil$ steps. Binary search on a sorted list of $n$ items needs at most

$$
\lfloor \log_2 n \rfloor + 1
$$

probes (comparisons with a middle element), because each probe either finds the target or discards the middle element and at least half of the rest. Doubling $n$ costs **one** extra step; multiplying $n$ by $1000 \approx 2^{10}$ costs about ten.

### 2.2 The base does not matter for $O(\log n)$

An algorithm that cuts its problem into thirds takes $\log_3 n$ steps; one that cuts into tenths takes $\log_{10} n$. By the change-of-base formula,

$$
\log_b n = \frac{\log_2 n}{\log_2 b},
$$

so all of these differ only by a **constant factor**. That is why computer scientists write $O(\log n)$ without a base. (When the base matters — counting exact steps — it is almost always $2$, and $\log n$ in a computer-science text usually means $\log_2 n$. In this notebook we always write the base.)

### 2.3 Divide and conquer: $T(n) = 2T(n/2) + n$

**Merge sort** sorts a list by splitting it in half, sorting each half recursively, and **merging** the two sorted halves, which takes about $n$ comparisons. Its cost satisfies

$$
T(n) = 2\,T(n/2) + n, \qquad T(1) = 0.
$$

Unroll it for $n = 2^k$:

$$
T(n) = n + 2\cdot\frac n2 + 4\cdot\frac n4 + \cdots = \underbrace{n + n + \cdots + n}_{k \text{ levels}} = n \log_2 n.
$$

The **recursion tree** has $\log_2 n$ levels (the halving again), and each level does a total of $n$ work. So the logarithm counts **levels**, and $n \log_2 n$ is levels times work per level. Compared with the $\approx n^2/2$ comparisons of simple methods such as insertion sort, this is an enormous saving: for $n = 10^6$, $n^2/2 = 5 \times 10^{11}$ against $n \log_2 n \approx 2 \times 10^{7}$.

$n \log_2 n$ is also essentially the best possible for sorting by comparisons: there are $n!$ possible orders, each comparison has two outcomes, so at least $\log_2 n! \approx n \log_2 n - 1.44\,n$ comparisons are needed in the worst case (exercise 7).

### 2.4 Fast exponentiation

Computing $x^{n}$ by multiplying $x$ by itself takes $n - 1$ multiplications. **Repeated squaring** uses the binary digits of $n$:

$$
x^{13} = x^{8 + 4 + 1} = x^{8} \cdot x^{4} \cdot x, \qquad x^2 = x \cdot x,\ x^4 = (x^2)^2,\ x^8 = (x^4)^2.
$$

Squarings: one per binary digit after the first, i.e. $\lfloor \log_2 n \rfloor$. Extra multiplications: one per additional $1$-bit. In total

$$
\lfloor \log_2 n \rfloor + (\text{number of 1-bits of } n) - 1 \;\le\; 2 \log_2 n
$$

multiplications. Python's built-in `pow(x, n, m)` works this way, and notebook 05 depends on it.

### 2.5 Balanced binary trees

A binary tree of height $h$ (the longest path from the root has $h$ edges) holds at most $1 + 2 + 4 + \cdots + 2^h = 2^{h+1} - 1$ nodes. So a tree with $n$ nodes has height

$$
h \ge \log_2(n + 1) - 1,
$$

and a perfectly balanced tree achieves $h = \lfloor \log_2 n \rfloor$. A search, insertion or deletion in a balanced search tree follows one root-to-leaf path: $O(\log n)$ steps. If the tree degenerates into a chain (for example by inserting keys in sorted order without rebalancing) the height becomes $n - 1$ and the logarithm is lost — which is why practical search trees (red–black trees, B-trees) rebalance themselves.

## 3. Worked Examples

### Example 1 — the phone book

$\log_2 10^9 = 9 \log_2 10 \approx 9 \times 3.3219 = 29.90$. Binary search needs at most $\lfloor 29.90 \rfloor + 1 = 30$ probes. Linear search needs up to $10^9$.

### Example 2 — binary search by hand

Search for $23$ in the sorted list $[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]$ ($16$ items, indices $0$–$15$):

| probe | range of indices | middle index | middle value | decision |
|---|---|---|---|---|
| 1 | $0$–$15$ | $7$ | $19$ | $23 > 19$: go right |
| 2 | $8$–$15$ | $11$ | $37$ | $23 < 37$: go left |
| 3 | $8$–$10$ | $9$ | $29$ | $23 < 29$: go left |
| 4 | $8$–$8$ | $8$ | $23$ | found |

Four probes; the bound is $\lfloor \log_2 16 \rfloor + 1 = 5$.

### Example 3 — unrolling the recurrence for $n = 8$

$T(1) = 0$, $T(2) = 2 \cdot 0 + 2 = 2$, $T(4) = 2 \cdot 2 + 4 = 8$, $T(8) = 2 \cdot 8 + 8 = 24$. And $8 \log_2 8 = 8 \times 3 = 24$ ✓.

### Example 4 — $3^{13}$ with five multiplications

$13 = 1101_2$. Squarings: $3^2 = 9$, $3^4 = 81$, $3^8 = 6561$ (three). Combine the $1$-bits: $3^8 \cdot 3^4 = 531441$, then $\cdot\, 3 = 1594323$ (two). Total $5 = \lfloor \log_2 13 \rfloor + 3 - 1$, instead of $12$.

### Example 5 — a balanced tree of a million keys

$\lfloor \log_2 10^6 \rfloor = \lfloor 19.93 \rfloor = 19$. A perfectly balanced search tree with a million keys has height $19$, so any lookup visits at most $20$ nodes.

## 4. Python Examples

### 4.1 Binary search with a probe counter

We count **probes** — comparisons of the target with a middle element — and check the worst case over every possible target against $\lfloor \log_2 n \rfloor + 1$.

In [ ]:
import math


def binary_search(sorted_items, target):
    # returns (index or None, number of probes)
    low, high = 0, len(sorted_items) - 1
    probes = 0
    while low <= high:
        middle = (low + high) // 2
        probes += 1
        if sorted_items[middle] == target:
            return middle, probes
        if sorted_items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return None, probes


primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]
print("search 23:", binary_search(primes, 23))
print("search 24:", binary_search(primes, 24))
print()

print(f"{'n':>7} {'worst probes':>13} {'mean probes':>12} {'floor(log2 n)+1':>16}")
for n in [10, 100, 1000, 10_000, 100_000]:
    items = list(range(n))
    probe_counts = [binary_search(items, target)[1] for target in items]
    print(f"{n:>7} {max(probe_counts):>13} {sum(probe_counts) / n:>12.2f} {math.floor(math.log2(n)) + 1:>16}")

The worst case matches the formula exactly, and the average is only one or two probes less: most items sit in the bottom levels of the implicit search tree.

### 4.2 The phone book of $10^9$ entries

A Python `range` can be indexed without storing its elements, so we can binary-search a "phone book" of a billion sorted numbers directly.

In [ ]:
import math


def binary_search(sorted_items, target):
    low, high = 0, len(sorted_items) - 1
    probes = 0
    while low <= high:
        middle = (low + high) // 2
        probes += 1
        if sorted_items[middle] == target:
            return middle, probes
        if sorted_items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return None, probes


phone_book = range(0, 2 * 10**9, 2)        # 10^9 sorted even numbers
print("entries:", len(phone_book))
for target in [0, 1_234_567_890, 1_999_999_998, 777]:
    index, probes = binary_search(phone_book, target)
    print(f"target {target:>13,}: index {index}, probes {probes}")
print("bound floor(log2 n) + 1 =", math.floor(math.log2(len(phone_book))) + 1)

### 4.3 Merge sort with a comparison counter

We count element comparisons and compare with $n \log_2 n$ and with the information-theoretic lower bound $\log_2 n!$ (computed as `math.lgamma(n + 1) / math.log(2)` to avoid the huge factorial).

In [ ]:
import math
import random


def merge_sort(items, counter):
    if len(items) <= 1:
        return items
    middle = len(items) // 2
    left = merge_sort(items[:middle], counter)
    right = merge_sort(items[middle:], counter)
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        counter[0] += 1
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


random.seed(0)
print(f"{'n':>8} {'comparisons':>12} {'n log2 n':>12} {'log2(n!)':>12} {'ratio to n log2 n':>18}")
for n in [16, 256, 4096, 65536]:
    data = [random.random() for _ in range(n)]
    counter = [0]
    result = merge_sort(data, counter)
    assert result == sorted(data)
    n_log_n = n * math.log2(n)
    lower_bound = math.lgamma(n + 1) / math.log(2)
    print(f"{n:>8} {counter[0]:>12} {n_log_n:>12.0f} {lower_bound:>12.0f} {counter[0] / n_log_n:>18.3f}")

Merge sort's count sits between the lower bound $\log_2 n!$ and $n \log_2 n$, and the ratio to $n \log_2 n$ creeps up towards $1$: on random data merge sort uses roughly $n \log_2 n - 1.25\,n$ comparisons, because a merge often finishes early when one half runs out.

### 4.4 Solving the recurrence exactly

For general $n$ the recurrence is $T(n) = T(\lfloor n/2 \rfloor) + T(\lceil n/2 \rceil) + n$. We tabulate it and compare with $n \log_2 n$; for powers of two they agree exactly.

In [ ]:
import math
from functools import lru_cache


@lru_cache(maxsize=None)
def recurrence_cost(n):
    if n <= 1:
        return 0
    return recurrence_cost(n // 2) + recurrence_cost(n - n // 2) + n


for n in [2, 8, 64, 1024, 1000, 10**6]:
    exact = recurrence_cost(n)
    estimate = n * math.log2(n)
    print(f"n = {n:>8}: T(n) = {exact:>10}, n log2 n = {estimate:>12.1f}, ratio {exact / estimate:.4f}")

### 4.5 Fast exponentiation with a multiplication counter

In [ ]:
import math


def fast_power(base, exponent):
    # right-to-left binary exponentiation; returns (result, multiplications)
    result = None
    square = base
    multiplications = 0
    while exponent > 0:
        if exponent % 2 == 1:
            if result is None:
                result = square
            else:
                result = result * square
                multiplications += 1
        exponent //= 2
        if exponent > 0:
            square = square * square
            multiplications += 1
    return result, multiplications


for n in [13, 45, 64, 1000, 10**6, 2**20 - 1]:
    value, count = fast_power(3, n)
    assert value == 3**n
    formula = math.floor(math.log2(n)) + bin(n).count("1") - 1
    print(f"n = {n:>8} = {bin(n):>23}: {count:>3} multiplications (formula {formula}), naive {n - 1}")

### 4.6 Tree heights: balanced, random and degenerate

We insert $n$ keys into a plain (non-rebalancing) binary search tree in three orders and measure the height. Insertion is written as a loop, not recursion, so the degenerate case does not hit Python's recursion limit.

In [ ]:
import math
import random


def build_search_tree_height(keys):
    # tree stored as dictionaries of child pointers; returns height (edges on longest root-to-leaf path)
    left_child, right_child = {}, {}
    root = keys[0]
    height = 0
    for key in keys[1:]:
        node, depth = root, 0
        while True:
            depth += 1
            children = left_child if key < node else right_child
            if node in children:
                node = children[node]
            else:
                children[node] = key
                break
        height = max(height, depth)
    return height


def balanced_order(sorted_keys):
    # middle first, then the middles of each half: produces a perfectly balanced tree
    order, stack = [], [(0, len(sorted_keys) - 1)]
    while stack:
        low, high = stack.pop()
        if low > high:
            continue
        middle = (low + high) // 2
        order.append(sorted_keys[middle])
        stack.append((low, middle - 1))
        stack.append((middle + 1, high))
    return order


random.seed(1)
print(f"{'n':>6} {'balanced':>9} {'random':>7} {'sorted':>7} {'floor(log2 n)':>14}")
for n in [15, 127, 1023, 4000]:
    keys = list(range(n))
    shuffled = keys[:]
    random.shuffle(shuffled)
    print(f"{n:>6} {build_search_tree_height(balanced_order(keys)):>9} {build_search_tree_height(shuffled):>7} "
          f"{build_search_tree_height(keys):>7} {math.floor(math.log2(n)):>14}")

Balanced insertion achieves $\lfloor \log_2 n \rfloor$ exactly; random insertion stays within a small multiple of $\log_2 n$; sorted insertion degenerates to a chain of height $n - 1$.

## 5. Visualizations

### 5.1 Binary search probes against $\log_2 n$

On a logarithmic $n$-axis, $\log_2 n$ is a straight line, and the worst-case probe count is a staircase hugging it from above.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np


def worst_case_probes(n):
    return math.floor(math.log2(n)) + 1


sizes = np.unique(np.logspace(0, 9, 400).astype(int))
probes = [worst_case_probes(n) for n in sizes]

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(sizes, probes, where="post", color="tab:blue", label="worst-case probes floor(log2 n) + 1")
ax.plot(sizes, np.log2(sizes), "--", color="crimson", label="log2 n")
ax.annotate("phone book: 10^9 entries, 30 probes", xy=(1e9, 30), xytext=(1e4, 27),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=10)
ax.set_xscale("log")
ax.set_xlabel("list length n (log scale)")
ax.set_ylabel("probes (linear scale)")
ax.set_title("Binary search: a billion items, thirty steps")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper left")
plt.show()

### 5.2 The recursion tree of merge sort

For $n = 16$: each level splits every block in two. There are $\log_2 16 = 4$ levels of merging, and each level handles all $16$ elements — total work $16 \times 4 = 64 = n \log_2 n$.

In [ ]:
import matplotlib.pyplot as plt

n = 16
levels = 5   # block sizes 16, 8, 4, 2, 1

fig, ax = plt.subplots(figsize=(11, 4.5))
for level in range(levels):
    block_size = n // 2**level
    for block in range(2**level):
        x_start = block * block_size
        ax.add_patch(plt.Rectangle((x_start + 0.08, -level - 0.4), block_size - 0.16, 0.8,
                                   facecolor="tab:blue", alpha=0.25 + 0.12 * level, edgecolor="black"))
        if block_size >= 2:
            ax.text(x_start + block_size / 2, -level, str(block_size), ha="center", va="center", fontsize=9)
    work_text = f"merge work {n}" if level < levels - 1 else "single elements: no work"
    block_word = "block" if level == 0 else "blocks"
    ax.text(n + 0.5, -level, f"level {level}: {2**level} {block_word} of {block_size} -> {work_text}",
            va="center", fontsize=9)

ax.set_xlim(0, n + 9)
ax.set_ylim(-levels + 0.4, 0.6)
ax.axis("off")
ax.set_title("Recursion tree: log2(16) = 4 levels of merging, n = 16 work per level")
plt.show()

### 5.3 Growth of sorting costs

Comparison counts of merge sort (measured) against $n$, $n \log_2 n$ and $n^2/2$ (insertion sort's typical cost), on log–log axes. Power laws are straight lines there; $n \log_2 n$ is a line of slope $1$ bent very slightly upward by the logarithm.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np


def merge_sort_comparisons(items):
    if len(items) <= 1:
        return items, 0
    middle = len(items) // 2
    left, left_count = merge_sort_comparisons(items[:middle])
    right, right_count = merge_sort_comparisons(items[middle:])
    merged, i, j, count = [], 0, 0, left_count + right_count
    while i < len(left) and j < len(right):
        count += 1
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged, count


random.seed(2)
sizes = np.unique(np.logspace(1, 4.7, 15).astype(int))
measured = [merge_sort_comparisons([random.random() for _ in range(n)])[1] for n in sizes]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(sizes, sizes**2 / 2, color="tab:red", label="n^2 / 2 (insertion sort)")
ax.plot(sizes, sizes * np.log2(sizes), color="tab:blue", label="n log2 n")
ax.plot(sizes, sizes, color="gray", linestyle="--", label="n")
ax.plot(sizes, measured, "o", color="black", markersize=5, label="merge sort, measured")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("n (log scale)")
ax.set_ylabel("comparisons (log scale)")
ax.set_title("Sorting cost: n log2 n versus n^2")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.show()

### 5.4 Multiplications for $x^n$

The number of multiplications in fast exponentiation, for every $n$ up to $2000$, lies between $\log_2 n$ and $2\log_2 n$; the spikes are at numbers just below a power of two (many $1$-bits), the dips at powers of two.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

exponents = np.arange(2, 2001)
multiplications = [math.floor(math.log2(n)) + bin(n).count("1") - 1 for n in exponents]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(exponents, multiplications, ".", markersize=2, color="tab:blue", label="repeated squaring (exact count)")
ax.plot(exponents, np.log2(exponents), color="green", label="log2 n")
ax.plot(exponents, 2 * np.log2(exponents), color="crimson", label="2 log2 n")
ax.set_xscale("log")
ax.set_xlabel("exponent n (log scale)")
ax.set_ylabel("multiplications")
ax.set_title("Computing x^n: about log2 n multiplications instead of n - 1")
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper left")
plt.show()

## 6. Exercises

### Basic

1. What is the maximum number of probes binary search needs on a sorted list of $1{,}000{,}000$ items?
2. Write $45$ in binary and count the multiplications repeated squaring needs for $x^{45}$.
3. What is the height of a perfectly balanced binary search tree holding $10^9$ keys?

### Intermediate

4. **Estimate it first, then compute.** You must sort $n = 10^6$ records on a machine doing $10^9$ comparisons per second. Estimate the time for an $n^2/2$ method and for an $n \log_2 n$ method, using $\log_2 10^6 \approx 20$. Then compute both.
5. Ternary search splits the range into three parts with two comparisons per step. Show that its worst case is about $2\log_3 n$ comparisons and compare with $\log_2 n$. Is it better or worse than binary search?
6. Solve the recurrences $A(n) = A(n/2) + 1$, $A(1) = 1$ and $B(n) = 2B(n/2) + 1$, $B(1) = 1$ for $n = 2^k$. Why does only the first one produce a logarithm?

### Advanced

7. A comparison sort can be drawn as a binary decision tree whose leaves are the $n!$ possible orderings. Explain why its height is at least $\log_2 n!$, and use Stirling's approximation $\ln n! \approx n \ln n - n$ to show this is about $n \log_2 n - 1.44\,n$. Compare with the merge-sort count for $n = 4096$ from section 4.3.
8. Prove by induction on $k$ that $T(n) = 2T(n/2) + n$, $T(1) = 0$ gives $T(2^k) = k\,2^k$, i.e. $T(n) = n \log_2 n$.
9. Prove that right-to-left repeated squaring uses exactly $\lfloor \log_2 n \rfloor + s(n) - 1$ multiplications, where $s(n)$ is the number of $1$-bits of $n$. Deduce that this is at most $2\lfloor \log_2 n \rfloor$.

## 7. Solutions

1. $\log_2 10^6 \approx 19.93$, so $\lfloor 19.93 \rfloor + 1 = 20$ probes.
2. $45 = 101101_2$: $\lfloor \log_2 45 \rfloor = 5$ squarings and $4$ one-bits, so $5 + 4 - 1 = 8$ multiplications (naively $44$).
3. $\lfloor \log_2 10^9 \rfloor = \lfloor 29.90 \rfloor = 29$.
4. Estimate: $n^2/2 = 5 \times 10^{11}$ comparisons $\approx 500$ s, about $8$ minutes; $n \log_2 n \approx 2 \times 10^7$ comparisons $\approx 0.02$ s. Exactly $n \log_2 n = 1.99 \times 10^7$, so $0.0199$ s; the ratio is about $25{,}000$.
5. Each step keeps a third, so $\log_3 n$ steps of $2$ comparisons: $2 \log_3 n = \frac{2}{\log_2 3} \log_2 n \approx 1.26 \log_2 n$. Worse than binary search by about $26\%$ — cutting into more pieces only helps if the extra comparisons are free.
6. $A(2^k) = A(2^{k-1}) + 1 = \dots = A(1) + k = k + 1 = \log_2 n + 1$. $B(2^k) = 2B(2^{k-1}) + 1$ gives $B(n) = 2n - 1$: linear. In $A$ each of the $\log_2 n$ levels costs $1$ and there is one branch; in $B$ the number of subproblems doubles each level, so the bottom level alone has $n$ of them and the work there dominates. A logarithm appears when the number of levels, not the number of leaves, sets the cost.
7. Each comparison has two outcomes, so after $h$ comparisons at most $2^h$ different outcomes can be distinguished. To sort every input correctly, the tree needs a distinct leaf for each of the $n!$ orderings, so $2^h \ge n!$ and $h \ge \log_2 n!$. With Stirling, $\log_2 n! = \frac{\ln n!}{\ln 2} \approx \frac{n \ln n - n}{\ln 2} = n \log_2 n - n \log_2 e \approx n \log_2 n - 1.443\,n$. For $n = 4096$ this is about $43{,}243$ (exact $\log_2 4096! \approx 43{,}250$); merge sort used $44{,}032$ in section 4.3, within $2\%$ of the bound.
8. $k = 0$: $T(1) = 0 = 0 \cdot 2^0$. If $T(2^{k}) = k\,2^{k}$, then $T(2^{k+1}) = 2T(2^k) + 2^{k+1} = 2k\,2^k + 2^{k+1} = (k + 1)\,2^{k+1}$ ✓.
9. $n$ has $L = \lfloor \log_2 n \rfloor + 1$ binary digits. The loop squares once between consecutive digits: $L - 1 = \lfloor \log_2 n \rfloor$ squarings. Each $1$-bit multiplies the accumulated result by the current square, except the first, which just initialises it: $s(n) - 1$ multiplications. Since $s(n) \le L$, the total is at most $2(L - 1) = 2\lfloor \log_2 n \rfloor$.

Numerical checks:

In [ ]:
import math
import random

print("Ex 1:", math.floor(math.log2(10**6)) + 1)
print("Ex 2:", bin(45), math.floor(math.log2(45)) + bin(45).count("1") - 1)
print("Ex 3:", math.floor(math.log2(10**9)))
n = 10**6
quadratic_seconds = n**2 / 2 / 1e9
n_log_n_seconds = n * math.log2(n) / 1e9
print("Ex 4:", quadratic_seconds, "s", round(n_log_n_seconds, 4), "s  ratio", round(quadratic_seconds / n_log_n_seconds))
print("Ex 5:", round(2 / math.log2(3), 4))


def recurrence_a(n):
    return 1 if n == 1 else recurrence_a(n // 2) + 1


def recurrence_b(n):
    return 1 if n == 1 else 2 * recurrence_b(n // 2) + 1


print("Ex 6:", [(2**k, recurrence_a(2**k), recurrence_b(2**k)) for k in range(0, 6)])

n = 4096
stirling_estimate = n * math.log2(n) - n * math.log2(math.e)
exact_bound = math.lgamma(n + 1) / math.log(2)
print("Ex 7:", round(stirling_estimate), round(exact_bound))


def merge_sort_count(items):
    if len(items) <= 1:
        return items, 0
    middle = len(items) // 2
    left, a = merge_sort_count(items[:middle])
    right, b = merge_sort_count(items[middle:])
    merged, i, j, count = [], 0, 0, a + b
    while i < len(left) and j < len(right):
        count += 1
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    return merged + left[i:] + right[j:], count


random.seed(0)
for size in [16, 256, 4096]:   # replay the random draws of section 4.3
    data = [random.random() for _ in range(size)]
    _, comparisons = merge_sort_count(data)
print("      merge sort comparisons for n = 4096 (same data as 4.3):", comparisons)


def tree_cost(n):
    return 0 if n == 1 else 2 * tree_cost(n // 2) + n


print("Ex 8:", all(tree_cost(2**k) == k * 2**k for k in range(0, 15)))
print("Ex 9:", all(math.floor(math.log2(m)) + bin(m).count("1") - 1 <= 2 * math.floor(math.log2(m))
                   for m in range(1, 100000)))

## 8. Conclusion

- **Halving** $n$ down to $1$ takes $\lceil \log_2 n \rceil$ steps: $2^k \ge n \iff k \ge \log_2 n$. Binary search needs at most $\lfloor \log_2 n \rfloor + 1$ probes — $30$ for a billion entries.
- The base of the logarithm only changes a constant factor ($\log_b n = \log_2 n / \log_2 b$), so complexity is written $O(\log n)$.
- **Divide and conquer** with linear combining work, $T(n) = 2T(n/2) + n$, gives $T(n) = n \log_2 n$: $\log_2 n$ levels of recursion, $n$ work per level. Merge sort achieves it, and $\log_2 n! \approx n \log_2 n - 1.44\,n$ shows no comparison sort can do essentially better.
- **Repeated squaring** computes $x^n$ with $\lfloor \log_2 n \rfloor + s(n) - 1 \le 2\log_2 n$ multiplications.
- A **balanced binary tree** with $n$ nodes has height $\lfloor \log_2 n \rfloor$; without balancing it can degrade to $n - 1$.
- Geometrically: on a log axis, $\log_2 n$ is a straight line — each doubling of the input adds one step.

**Next:** notebook 05 uses fast exponentiation as a weapon. Computing $g^x \bmod p$ is easy (a few hundred multiplications even for huge numbers), but finding $x$ from $g^x \bmod p$ — the **discrete logarithm** — appears to be very hard, and that asymmetry lets two strangers agree on a secret over a public channel.